
Instalação das bibliotecas necessárias

---


*   Langchain > Orquestrador: Encadeia etapas de processamento, como receber uma pergunta, buscar dados, montar o contexto e gerar a resposta.

*   ChromaDB > anco de dados vetorial de código aberto (licença Apache 2.0) criado para armazenar, gerenciar e consultar embeddings

*   pypdf > Biblioteca para ler, mapnipular e modificar arquivos PDF





In [ ]:
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-groq>=0.3,<0.4" chromadb pypdf sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.19 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
langgraph-sdk 0.4.12 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.


In [ ]:
# Importações básicas
import os

# loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Banco vetorial
from langchain_community.vectorstores import Chroma

# Cadeia RAG
from langchain.chains import RetrievalQA

Lendo o arquivo "Analise de mercado" em PDF

In [ ]:
# Caminho do PDF com a análise de mercado
CAMINHO_PDF = "/content/Analise de mercado 1° TRI ABF.pdf"

# Carrega o PDF
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)

23

Criando os vetores, transformando o texto do PDF em CHUNKS

(pedações de texto que estarão na Vector Store)

In [ ]:
# Divide os documentos em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

len(chunks)

31

Criando a camada de vetores numéricos e banco vetorial

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Inicializa embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Cria o banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_abf_franchising"
)

/tmp/ipykernel_938/62739235.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Recuperação dos 3 argumentos mais próximos do PROMPT

In [ ]:
# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
#Criando conexão com o modelo LLAMA hospedado no GROQ, "Temperatura = 0 deixa mais determinístico"
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=userdata.get("GROQ_API_KEY"),
    temperature=0
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

Melhorando a qualidade das respostas, pois as inmstruções não dizem ao modelo para se limitar apenas ao contexto, fazendo com que ele responda fora da janela de informações.

In [ ]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Responda em português, usando apenas o contexto abaixo.
Se a resposta não estiver no contexto, diga que não encontrou essa informação no documento.

Contexto:
{context}

Pergunta: {question}

Resposta:"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
resposta = qa_chain.invoke({"query": "Qual segmento teve maior crescimento e de quanto?"})

print(resposta["result"])

for doc in resposta["source_documents"]:
    print("Página:", doc.metadata.get("page")+1)

O segmento **Alimentação – Comercialização e Distribuição** teve o maior crescimento, com uma variação de **+22,0 %** no primeiro trimestre de 2026 em relação ao mesmo período de 2025.
Página: 16
Página: 12
Página: 5
